In [ ]:
%load_ext autoreload
%autoreload 2
# Import required libraries
import sys
sys.path.append("../")
import os
import numpy as np
import utm
import io
import shutil
import h5py
import json
import matplotlib.pyplot as plt
from PIL import Image
from typing import List
import cv2

# Load customized utility functions
from utils.transformations import rotation_matrix_from_angles, get_yaw_pitch_roll, filter_above_ground, compute_heading
from utils.plot import plot_lidar_camera_oxts
from utils.process import memory_usage

# Load ZOD DevKit
from zod import ZodFrames, ZodSequences, ZodDrives
import zod.constants as constants
from zod.constants import Camera, Lidar, Anonymization, AnnotationProject
from zod.data_classes.ego_motion import OXTS_TIMESTAMP_OFFSET, interpolate_transforms
from zod.data_classes import LidarData
from zod.utils.geometry import transform_points
from zod.visualization.lidar_on_image import get_3d_transform_camera_lidar
from zod.constants import Camera, Lidar, Anonymization


# Load aerial image download functions
test_country = 'DK'

if test_country == 'SE': # can be used as training 
    from Map_Downloading.scripts.sweden import sweden_exact_position_image as download_sat
    from Map_Downloading.scripts.sweden import RESOLUTION as resolution
elif test_country == 'FR': # can be used as training
    from Map_Downloading.scripts.france import france_exact_position_image as download_sat
    from Map_Downloading.scripts.france import TRUE_RESOLUTION as resolution
elif test_country == 'BE': # can be used as training
    from Map_Downloading.scripts.belgium import belgium_exact_position_image as download_sat
    from Map_Downloading.scripts.belgium import RESOLUTION as resolution
elif test_country == 'NL': # can be used as training 
    from Map_Downloading.scripts.netherlands import nl_exact_position_image as download_sat
    from Map_Downloading.scripts.netherlands import RESOLUTION as resolution
elif test_country == 'DK': # can be used as training 
    from Map_Downloading.scripts.denmark import denmark_exact_position_image as download_sat
    from Map_Downloading.scripts.denmark import TILE_MATRIX_INFO
    tile_matrix_info = TILE_MATRIX_INFO['15']
    resolution = tile_matrix_info["ScaleDenominator"] * 0.28e-3

# Set matplotlib settings
%matplotlib inline

# NOTE! Set the path to dataset and choose a version
# dataset_root = "/Users/ziminxia/Work/Collaborations/Zenseact/data"  # your local path to zod
dataset_root = "/work/vita/datasets/zod"  # your local path to zod
version = "full"  # "mini" or "full"


# Load and display sensor frames
sensor_frames = plt.imread('./sensor_frames.png')
plt.figure(figsize=(20, 10))
plt.imshow(sensor_frames)
plt.axis('off')
plt.show()


In [ ]:
# initialize ZodSequences
zod_sequences = ZodSequences(dataset_root=dataset_root, version=version)

# get default training and validation splits
training_sequences = zod_sequences.get_split(constants.TRAIN)
validation_sequences = zod_sequences.get_split(constants.VAL)

print(f"Number of training sequences: {len(training_sequences)}")
print(f"Number of validation sequences: {len(validation_sequences)}")

all_sequence_num = list(training_sequences) + list(validation_sequences)
print(f"Number of total sequences: {len(all_sequence_num)}")

sequences_per_country = {}

for i in range(len(all_sequence_num)):
    zod_sequence = zod_sequences[all_sequence_num[i]]
    metadata = zod_sequence.metadata
    
    if metadata.country_code in sequences_per_country:
        sequences_per_country[metadata.country_code].append(all_sequence_num[i])
    else:
        sequences_per_country[metadata.country_code] = [all_sequence_num[i]]
        
for country in sequences_per_country.keys():
    print(country, len(sequences_per_country[country]))

In [ ]:
size = 100 # side length (m) of the requested aerial image


# List of drives to process

for sequence_idx in sequences_per_country[test_country]:

    # Define Paths
    sequence_path = os.path.join(dataset_root, 'sequences', sequence_idx)
    aerial_path = os.path.join(sequence_path, "aerial_img")
    ground_path = os.path.join(sequence_path, "ground_img")

    # # Create or Reset Directories
    # for path in [aerial_path, ground_path]:
    #     if os.path.exists(path):
    #         shutil.rmtree(path)
    #         print(f"Removed existing folder: {path}")
    #     os.makedirs(path)
    #     print(f"Created new folder: {path}")

    sequence = zod_sequences[sequence_idx]
    calibrations = sequence.calibration

    cam_intrinsics_4x3 = calibrations.cameras[Camera.FRONT].intrinsics
    cam_intrinsics = cam_intrinsics_4x3[:, :3]
    cam_distortion = calibrations.cameras[Camera.FRONT].distortion
    image_dimensions = tuple(calibrations.cameras[Camera.FRONT].image_dimensions)

    K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
            cam_intrinsics, cam_distortion, image_dimensions, np.eye(3), balance=0.0
        )
        
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(
        cam_intrinsics, cam_distortion, np.eye(3), K_new, image_dimensions, cv2.CV_16SC2
    )

    T_cam2oxts = calibrations.get_extrinsics(Camera.FRONT).transform
    T_lidar2oxts = calibrations.lidars[Lidar.VELODYNE].extrinsics.transform

    num_frames = len(sequence.info.camera_frames['front_blur'])
    filename = os.path.join(sequence_path, "oxts.hdf5")

    # Read HDF5 Data Once
    with h5py.File(filename, "r") as f:
        lat = f['posLat'][()]
        lon = f['posLon'][()]
        alt = f['posAlt'][()]
        yaw = 90 - f['heading'][()] 
        if sequence.metadata.country_code == 'SE':
            yaw = yaw - 2.5  # Adjusted heading
        pitch = f['pitch'][()]
        roll = f['roll'][()]
        timestamp = OXTS_TIMESTAMP_OFFSET + f['timestamp'][()] + f['leapSeconds'][()][0]

    heading_difference = []
    T_prev = np.eye(4)

    for i in range(0, num_frames):
        if i % 100 == 0:
            print(f"Frame {i}/{num_frames}, Memory Usage: {memory_usage():.2f} MB")

        current_timestamp = sequence.info.camera_frames['front_blur'][i].time.timestamp()
        
        # Use np.searchsorted for efficient timestamp lookup
        oxts_idx = np.searchsorted(timestamp, current_timestamp, side='right')

        if oxts_idx == 0 or oxts_idx >= len(timestamp):
            continue  # Skip if no valid index

        # Extract previous and next OXTS readings
        lat_prev, lon_prev, alt_prev = lat[oxts_idx - 1], lon[oxts_idx - 1], alt[oxts_idx - 1]
        yaw_prev, pitch_prev, roll_prev = yaw[oxts_idx - 1], pitch[oxts_idx - 1], roll[oxts_idx - 1]
        easting_prev, northing_prev, zone_number_prev, zone_letter_prev = utm.from_latlon(lat_prev, lon_prev)

        lat_next, lon_next, alt_next = lat[oxts_idx], lon[oxts_idx], alt[oxts_idx]
        yaw_next, pitch_next, roll_next = yaw[oxts_idx], pitch[oxts_idx], roll[oxts_idx]
        easting_next, northing_next, zone_number_next, zone_letter_next = utm.from_latlon(lat_next, lon_next)

        # Compute previous and next transformation matrices
        T_previous = np.eye(4)
        T_previous[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_prev), np.radians(pitch_prev), np.radians(yaw_prev), order="ZYX"
        )
        T_previous[:3, 3] = [easting_prev, northing_prev, alt_prev]

        T_next = np.eye(4)
        T_next[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_next), np.radians(pitch_next), np.radians(yaw_next), order="ZYX"
        )
        T_next[:3, 3] = [easting_next, northing_next, alt_next]

        # Interpolate transformation
        interp_factor = (current_timestamp - timestamp[oxts_idx - 1]) / (timestamp[oxts_idx] - timestamp[oxts_idx - 1])
        T_current = interpolate_transforms(T_previous, T_next, interp_factor)
        yaw_oxts, _, _ = get_yaw_pitch_roll(T_current)
        heading_oxts = 90 - np.degrees(yaw_oxts)

        # Save Ground Image
        grd_image = sequence.info.camera_frames['front_blur'][i].read()
        grd_image = cv2.remap(grd_image, map1, map2, interpolation=cv2.INTER_LINEAR)
        # Image.fromarray(grd_image).resize((grd_image.shape[1] // 4, grd_image.shape[0] // 4), Image.LANCZOS).save(
        #     os.path.join(ground_path, f'frame{i:06}.png'), format="PNG"
        # )
        plt.imshow(grd_image)
        plt.show()

        # Process Lidar Data
        pcd = sequence.get_compensated_lidar(sequence.info.camera_frames['front_blur'][i].time).points
        pcd_utm = transform_points(pcd, T_current @ T_lidar2oxts)
        pcd_utm = filter_above_ground(pcd_utm, ground_threshold=1)

        # Compute heading difference
        heading_move = compute_heading(T_prev[0, 3], T_prev[1, 3], T_current[0, 3], T_current[1, 3])

        if i > 0:
            difference = (heading_move - heading_oxts + 180) % 360 - 180
            heading_difference.append(difference)

        T_prev = T_current.copy()

        # Retrieve Aerial Image
        lat_current, lon_current = utm.to_latlon(T_current[0, 3], T_current[1, 3], zone_number_next, zone_letter_next)
        aerial_img = download_sat(lat_current, lon_current, size)

        # Plot BEV (Bird’s Eye View) with LiDAR points
        bev_image = np.array(aerial_img)
        H, W = bev_image.shape[:2]
        bev_center = (T_current[0, 3], T_current[1, 3])

        # Convert LiDAR points to BEV pixel coordinates
        x_indices = ((pcd_utm[:, 0] - bev_center[0]) / resolution + W / 2).astype(int)
        y_indices = H - ((pcd_utm[:, 1] - bev_center[1]) / resolution + H / 2).astype(int)
        x_indices, y_indices = np.clip(x_indices, 0, W - 1), np.clip(y_indices, 0, H - 1)

        plt.figure(figsize=(8, 8))
        plt.imshow(bev_image, origin='upper')
        plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
        plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_move)), np.cos(np.radians(heading_move)), color='cyan', scale=20, label='Movement')
        plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_oxts)), np.cos(np.radians(heading_oxts)), color='r', scale=20, label='OXTS')
        plt.legend(loc=2)
        plt.axis('off')
        # plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
        plt.show()
        plt.close()

    print(f"Sequence {sequence_idx} Completed. Heading Difference Mean: {np.mean(heading_difference):.2f}, Median: {np.median(heading_difference):.2f}")